# Capstone Project — Architect, Evaluate, and Govern an Enterprise Claude Solution End-to-End

**Level: Intermediate**

## Objective

Act as the lead architect for a realistic enterprise engagement and deliver a complete, defensible Claude solution: conduct discovery, choose an architecture pattern with a documented ADR, design a RAG + integration layer with authentication/authorization and observability, build an evaluation harness (metrics + dataset + A/B), model cost/latency trade-offs, apply guardrails, compliance (HIPAA/GDPR) and human-in-the-loop, and produce a stakeholder-facing architecture document plus a lifecycle/handoff plan. This capstone integrates every domain of the CCAR-P exam.

### What You'll Build

**Scenario — MedPlan Health.** A national health insurer wants to triage **prior-authorization** requests (clinical approve/deny/needs-info recommendations grounded in policy documents). It is high-stakes (PHI/HIPAA, nurse must approve), high-volume (20k requests/day), and bound by SLA (p95 < 20s). You will produce a set of artifacts — discovery doc, ADR, architecture spec, RAG chunker, authZ gap analysis, observability schema, eval harness, A/B result, cost model, guardrail + compliance matrices, HITL policy, and a final architecture document — all written to files in this folder.

### Rules

- Reuse the patterns and artifacts you built in Phases 1-5; reference lessons by number.
- Every live API cell must be guarded so the notebook runs with NO ANTHROPIC_API_KEY (mock fallback).
- Prefer decision artifacts (rubrics, matrices, trade-off tables) over prose — this exam tests judgment.
- Verify any Claude-specific syntax, model IDs, and pricing against https://docs.anthropic.com.
- Do the work yourself before expanding the hint code; the # TODO markers are where you decide.

## Prerequisites

Before starting, make sure you have:

- [ ] Completed Phases 1-5 of this study plan
- [ ] Python 3 kernel; `pip install anthropic` (optional — mock fallback provided)
- [ ] The CCAR-P exam guide, Domains 1-7, reviewed
- [ ] ~4-6 hours of focused time

**Estimated time:** 4-6 hours

In [ ]:
# !pip install anthropic
import os, json, math, random

try:
    import anthropic
    if os.environ.get("ANTHROPIC_API_KEY"):
        client = anthropic.Anthropic(); LIVE = True
    else:
        client = None; LIVE = False
except Exception as e:
    client = None; LIVE = False
    print("anthropic SDK unavailable:", e)

MODELS = {"opus":"claude-opus-4-20250514","sonnet":"claude-sonnet-4-20250514","haiku":"claude-3-5-haiku-20241022"}

def ask_claude(prompt, model=MODELS["sonnet"], system=None, max_tokens=512):
    if not LIVE:
        return f"[MOCK {model}] would answer: {prompt[:80]}..."
    msg = client.messages.create(model=model, max_tokens=max_tokens,
        system=system or "You are a senior solution architect.",
        messages=[{"role":"user","content":prompt}])
    return msg.content[0].text

print("LIVE" if LIVE else "MOCK", "mode ready — all cells run without an API key")

## Step 1: Discovery and Solution Brief

**Lesson reference:** Lessons 1.1, 5.4

Run structured discovery for MedPlan and frame the solution. Capture the measurable success metric, constraints, and value-pillar scoring, then write a solution brief. This anchors every later decision — if the success metric is not measurable, stop and fix it.

In [ ]:
discovery = {
  "business_goal": "Cut prior-auth handle time while keeping clinical quality",
  "kpi": "40% less handle time; >=95% agreement with nurses on a gold set",
  "users": "Utilization-review nurses; ~20,000 requests/day",
  "data": "Policy PDFs, member plan docs, clinical criteria (contains PHI)",
  "sla": "p95 < 20s; 99.9% uptime",
  "compliance": "HIPAA; some EU members -> GDPR",
  "risk": "Wrong denial harms patients; human nurse must approve every decision",
}
pillars = {"efficiency":5,"transformation":2,"productivity":4,"cost":3,"performance":4}
brief = ["# Solution Brief: MedPlan prior-auth triage",""]
brief += [f"- **{k}**: {v}" for k,v in discovery.items()]
brief += ["","## Value pillars"] + [f"- {k}: {v}/5" for k,v in pillars.items()]
open("cap_solution_brief.md","w",encoding="utf-8").write("\n".join(brief))
print("\n".join(brief))
# TODO: tighten the success metric and add at least one constraint discovery missed

### Checkpoint 1: Verify Discovery and Solution Brief

The brief must contain a measurable KPI and explicit HIPAA + HITL constraints.

In [ ]:
txt = open("cap_solution_brief.md",encoding="utf-8").read()
assert "95%" in txt and "HIPAA" in txt
assert "nurse" in txt.lower() or "human" in txt.lower()
print("Checkpoint 1 PASS: measurable KPI + HIPAA + HITL present")

## Step 2: Pattern Selection and ADR

**Lesson reference:** Lessons 1.3, 1.5

Choose the architecture pattern with a rubric, then record the decision as an ADR. For a high-stakes, auditable, SLA-bound flow, a workflow pattern usually beats an agentic loop — but justify it from the traits, do not assume.

In [ ]:
def select_pattern(dynamic, subtasks, auditability_critical, latency_budget_s):
    if dynamic and latency_budget_s >= 10 and not auditability_critical:
        return "agentic"
    if subtasks >= 2:
        return "workflow"
    return "augmented_llm"

pattern = select_pattern(dynamic=False, subtasks=4, auditability_critical=True, latency_budget_s=20)
print("selected pattern:", pattern)
adr = f'''# ADR-001: Architecture pattern for MedPlan prior-auth triage

## Status
Accepted

## Context
High-stakes HIPAA decisions, p95<20s, nurse approval required.

## Decision
Use a **{pattern}**: redact -> retrieve policy -> draft -> validate citations -> human review.

## Alternatives
- Augmented LLM: too little control for multi-step validation.
- Agentic loop: unneeded flexibility hurts auditability/latency/cost.

## Consequences
+ Auditable, each step evaluable.
- New request shapes need code changes.
'''
open("cap_ADR-001.md","w",encoding="utf-8").write(adr)
print(adr[:300])
# TODO: write cap_ADR-002.md for the model-tiering decision (Haiku route vs Sonnet draft)

### Checkpoint 2: Verify Pattern Selection and ADR

ADR-001 exists with all four sections and a pattern justified by the rubric.

In [ ]:
a = open("cap_ADR-001.md",encoding="utf-8").read()
assert all(s in a for s in ["## Context","## Decision","## Alternatives","## Consequences"])
print("Checkpoint 2 PASS: ADR-001 complete")

## Step 3: RAG + Integration Layer with AuthN/AuthZ

**Lesson reference:** Lessons 3.1, 3.2, 3.5, 3.6

Design the retrieval and integration layer: chunk policy docs, pick a retrieval strategy, run a least-privilege tool audit, and close authZ gaps so retrieval is scoped to the member. Unscoped retrieval leaks PHI across members — the critical failure to prevent.

In [ ]:
def chunk_document(text, chunk_tokens=200, overlap_tokens=40):
    words=text.split(); wpc=int(chunk_tokens*0.75); wov=int(overlap_tokens*0.75)
    step=max(1,wpc-wov); out=[]; i=0
    while i<len(words): out.append(" ".join(words[i:i+wpc])); i+=step
    return out

policy=("MRI requires prior authorization after a failed six week course of physical therapy. "
        "Emergency imaging is exempt from prior authorization. ")*12
chunks=chunk_document(policy,60,15)
retrieval="hybrid (BM25 + dense) + re-ranking"  # policy codes + NL questions, high stakes
tools={"read_request":["nurse"],"policy_search":["nurse"],"member_lookup":["nurse"],
       "issue_payment":["billing"],"delete_member":["admin"]}
role="nurse"
keep=[t for t,r in tools.items() if role in r]; remove=[t for t in tools if role not in tools[t]]
hops=[{"hop":"agent->vector_store","authn":True,"authz":True,"scoped_to_member":True}]
gaps=[h['hop'] for h in hops if not h['scoped_to_member']]
print(f"chunks={len(chunks)} retrieval={retrieval}")
print("KEEP tools:",keep,"| REMOVE (bloat):",remove)
print("authZ gaps:", gaps or "none — retrieval scoped to member")
# TODO: add a protocol decision (MCP vs API/CLI vs A2A) for the policy-search integration and justify it

### Checkpoint 3: Verify RAG + Integration Layer with AuthN/AuthZ

High-risk tools removed for the nurse role and retrieval is member-scoped.

In [ ]:
assert "delete_member" in remove and "issue_payment" in remove
assert not gaps, "retrieval must be scoped to the member (no PHI cross-leak)"
print("Checkpoint 3 PASS: least privilege + scoped retrieval")

## Step 4: Evaluation Harness: Metrics, Dataset, A/B

**Lesson reference:** Lessons 4.1, 4.2, 4.3

Build the eval harness that proves quality: a labeled dataset with edge cases, multi-dimensional metrics, and an A/B comparison of two variants with a significance check. This is the artifact that earns the right to ship changes.

In [ ]:
dataset=[{"id":"c1","input":"MRI after 8wk failed PT","gold":"approve"},
         {"id":"c2","input":"MRI, no PT","gold":"deny"},
         {"id":"c3","input":"MRI, PT undocumented","gold":"needs_info"},
         {"id":"c4","input":"Emergency CT after trauma","gold":"approve"}]
open("cap_eval_dataset.jsonl","w",encoding="utf-8").write("\n".join(json.dumps(r) for r in dataset))
def accuracy(preds,gold): return sum(p==g for p,g in zip(preds,gold))/len(gold)
import math,random; random.seed(1)
n=60
A=[1 if random.random()<0.85 else 0 for _ in range(n)]  # control
B=[1 if random.random()<0.92 else 0 for _ in range(n)]  # new prompt variant
def z(a,b):
    na,nb=len(a),len(b); pa,pb=sum(a)/na,sum(b)/nb; p=(sum(a)+sum(b))/(na+nb)
    se=math.sqrt(p*(1-p)*(1/na+1/nb)) or 1e-9; return (pb-pa)/se
zz=z(A,B)
print(f"A acc={sum(A)/n:.3f} B acc={sum(B)/n:.3f} z={zz:.2f} significant={abs(zz)>1.96}")
# TODO: add latency, cost, safety, and security metrics and an overall pass/fail gate

### Checkpoint 4: Verify Evaluation Harness: Metrics, Dataset, A/B

Dataset persisted and A/B produces a decision signal.

In [ ]:
lines=open("cap_eval_dataset.jsonl",encoding="utf-8").read().strip().splitlines()
assert len(lines)>=4
assert isinstance(zz,float)
print("Checkpoint 4 PASS: eval dataset + A/B harness working")

## Step 5: Cost / Latency Trade-off Model

**Lesson reference:** Lessons 3.3, 4.5

Model the cost and latency of candidate configurations and select the one that meets the SLA and accuracy floor at lowest cost. Recommend a config and state what you traded away.

In [ ]:
configs=[{"name":"haiku+RAG","acc":0.86,"p95_ms":900,"cost":1},
         {"name":"sonnet+RAG","acc":0.92,"p95_ms":1800,"cost":3},
         {"name":"sonnet+RAG+CoT","acc":0.95,"p95_ms":2600,"cost":4},
         {"name":"opus+multiagent","acc":0.97,"p95_ms":6000,"cost":12}]
def choose(configs,min_acc,sla_ms):
    feas=[c for c in configs if c['acc']>=min_acc and c['p95_ms']<=sla_ms]
    return min(feas,key=lambda c:c['cost']) if feas else None
pick=choose(configs,min_acc=0.90,sla_ms=20000)
print("chosen config:",pick)
# TODO: justify in one line; note which accuracy you gave up vs opus+multiagent and why it's acceptable

### Checkpoint 5: Verify Cost / Latency Trade-off Model

A feasible config meeting accuracy>=0.90 within the 20s SLA is selected.

In [ ]:
assert pick and pick['acc']>=0.90 and pick['p95_ms']<=20000
print("Checkpoint 5 PASS: SLA-feasible config chosen:", pick['name'])

## Step 6: Guardrails, Compliance, and Human-in-the-Loop

**Lesson reference:** Lessons 5.1, 5.2, 5.3

Apply the governance wrapper: a layered guardrail checklist, a HIPAA/GDPR control matrix, a fairness-by-slice check, and a HITL routing policy that sends high-impact or low-confidence decisions to a nurse. Fairness gaps and PHI handling are non-negotiable.

In [ ]:
guardrails={"input":["injection detection","PHI redaction"],
            "output":["schema validation","citation check","PHI scrub"],
            "human":["nurse review of all final decisions"]}
compliance={"HIPAA":["access controls","audit log","BAA","encryption"],
            "GDPR":["lawful basis","erasure","EU residency","human review of automated decisions"]}
json.dump({"guardrails":guardrails,"compliance":compliance},
          open("cap_governance.json","w",encoding="utf-8"),indent=2)
slices=[("group_A",92,100),("group_B",88,100),("group_C",90,100)]
accs={g:c/t for g,c,t in slices}; gap=max(accs.values())-min(accs.values())
def hitl(impact,conf,irreversible):
    if irreversible or impact>=4 or conf<0.7: return "human_review_required"
    return "auto_proceed" if conf>=0.85 else "human_spot_check"
print("fairness gap=",round(gap,2),"->","OK" if gap<=0.05 else "CONCERN")
print("HITL (impact5,conf0.95,irrev):",hitl(5,0.95,True))
# TODO: add a data_flow_check that blocks PHI to a non-BAA vendor and EU data leaving the EU

### Checkpoint 6: Verify Guardrails, Compliance, and Human-in-the-Loop

Governance file written with HIPAA controls, HITL for high-impact, and fairness within tolerance.

In [ ]:
g=json.load(open("cap_governance.json",encoding="utf-8"))
assert "HIPAA" in g['compliance'] and gap<=0.05
assert hitl(5,0.95,True)=="human_review_required"
print("Checkpoint 6 PASS: guardrails + compliance + HITL in place")

## Step 7: Architecture Document and Lifecycle/Handoff Plan

**Lesson reference:** Lessons 1.2, 5.5

Assemble the stakeholder-facing architecture document and the lifecycle/handoff plan that indexes every artifact you produced. This is the deliverable the organization owns and operates — the capstone's final output.

In [ ]:
doc=["# Architecture: MedPlan prior-auth triage","",
     "## Overview\nInput(redact) -> processing(hybrid RAG + Claude) -> output(validated+cited) -> feedback(eval + nurse HITL)","",
     "## SLAs\np95<20s; >=95% nurse agreement; 99.9% uptime","",
     "## Decisions\n- cap_ADR-001.md (pattern)\n- cap_ADR-002.md (model tiering)","",
     "## Controls\n- least-privilege tools\n- layered guardrails\n- HIPAA/GDPR matrix\n- HITL on high-impact","",
     "## Artifacts\n- cap_solution_brief.md\n- cap_eval_dataset.jsonl\n- cap_governance.json","",
     "## Runbook\n- retrieval-miss spike -> inspect index (Sample 3)\n- accuracy drop -> run eval set\n- rollback -> pin prompt_version"]
open("cap_architecture_doc.md","w",encoding="utf-8").write("\n".join(doc))
lifecycle=[("discovery","signed brief+SLAs"),("design","ADRs+arch doc+eval plan"),
           ("handoff","doc+harness+dashboards+runbook"),("monitoring","alerts live"),
           ("iteration","A/B-validated improvements")]
for ph,exit in lifecycle: print(f"{ph:11} exit: {exit}")
print("\narchitecture doc written.")
# TODO: add a handoff checklist (each item verified) that must be fully True before go-live

### Checkpoint 7: Verify Architecture Document and Lifecycle/Handoff Plan

The architecture doc references SLAs, decisions, controls, and a runbook.

In [ ]:
d=open("cap_architecture_doc.md",encoding="utf-8").read()
assert "SLAs" in d and "Runbook" in d and "ADR-001" in d
print("Checkpoint 7 PASS: architecture document + lifecycle plan complete")

## Verification Checklist

Confirm each item before considering the capstone complete:

- [ ] Discovery brief with a measurable KPI, HIPAA, and HITL constraints (cap_solution_brief.md)
- [ ] ADR-001 (and ADR-002) with Context/Decision/Alternatives/Consequences
- [ ] RAG chunker + retrieval strategy chosen for the data/query shape
- [ ] Least-privilege tool audit removing high-risk tools for the nurse role
- [ ] AuthZ gap analysis proving retrieval is member-scoped (no PHI cross-leak)
- [ ] Eval dataset (JSONL) with edge cases + multi-dimensional metrics
- [ ] A/B comparison with a significance check and a ship/no-ship decision
- [ ] Cost/latency model selecting an SLA-feasible config at lowest cost
- [ ] Layered guardrails + HIPAA/GDPR matrix + fairness-by-slice check
- [ ] HITL policy routing high-impact/low-confidence to a human
- [ ] Stakeholder architecture document + lifecycle/handoff plan
- [ ] Every notebook cell runs with NO ANTHROPIC_API_KEY set (mock fallback)

## Stretch Goals

Completed the core project? Try these extensions:

1. **Multi-agent variant** — Replace the drafting step with an orchestrator-worker design (Lesson 1.4) and A/B it against the workflow on accuracy, latency, and cost.
2. **Prompt caching savings** — Model the cost impact of caching the static policy prefix (Lesson 2.4) and quantify the break-even call volume.
3. **Live wiring** — With a real key, wire the RAG retriever to an actual vector store and run the eval dataset end-to-end, recording real latency/cost.
4. **Progressive discovery** — Convert the monolithic tool list to a progressive-discovery mechanism (Lesson 3.6) and measure context-token savings.
5. **Red-team the guardrails** — Write 10 adversarial/injection inputs and report the input-guardrail catch rate; patch any misses.

---

## Congratulations! 🎉

You have architected, evaluated, and governed a production-grade Claude solution end-to-end — the exact competency the Claude Certified Architect – Professional (CCAR-P) exam measures across all seven domains. You produced a traceable chain of artifacts from discovery to handoff, each defensible to a stakeholder, a security reviewer, and a compliance officer.

You've demonstrated proficiency in:

- Translating a business problem into a measurable, scoped Claude solution (Domain 1)
- Selecting models, prompts, and context strategies by trade-off (Domain 2)
- Designing secure, observable RAG + integration layers (Domain 3)
- Building evaluation, A/B, and optimization harnesses (Domain 4)
- Applying guardrails, compliance, and human-in-the-loop governance (Domain 5)
- Communicating decisions and managing the solution lifecycle (Domain 6)
- Enabling teams with Claude tooling and operational support (Domain 7)

**What's next?**

- Review any lessons you found challenging
- Attempt the stretch goals above
- Book your exam when you can complete this capstone unaided